# Tạo và Nạp dữ liệu cho bảng FACT_ORDER_ITEMS
Mỗi dòng của bảng là một sản phẩm trong một đơn hàng:
- Số lượng mua (`purchased_quantity`)
- Số lượng trả (`returned_quantity`)
- Khóa ngoại đến `DIM_DATE`, `DIM_GEOGRAPHY` và `DIM_PRODUCT`.
Các giá trị tiền và khuyến mãi không được đưa vào fact này vì phạm vi hiện tại chỉ phân tích số lượng.

In [1]:
import os
from pathlib import Path

import pandas as pd
from db_engine import get_engine

engine = get_engine()

dim_geography = pd.read_sql(
    "SELECT geography_key, zip FROM dbo.dim_geography",
    engine
)
dim_product = pd.read_sql(
    "SELECT product_key, product_id FROM dbo.dim_product",
    engine
)
dim_geography["zip"] = dim_geography["zip"].astype("string")
dim_product["product_id"] = dim_product["product_id"].astype("string")


In [2]:
order_items = pd.read_csv(
    "../../SilverData/order_items_silver.csv",
    dtype={
        "item_id": "string",
        "order_id": "string",
        "product_id": "string"
    }
)
orders = pd.read_csv(
    "../../SilverData/orders_silver.csv",
    usecols=["order_id", "order_date", "zip"],
    dtype={
        "order_id": "string",
        "zip": "string"
    }
)
returns = pd.read_csv(
    "../../SilverData/returns_silver.csv",
    dtype={
        "order_id": "string",
        "product_id": "string"
    }
)

returned = returns.groupby(
    ["order_id", "product_id"],
    as_index=False
).agg(
    returned_quantity=("return_quantity", "sum")
)

df_items = order_items.merge(
    returned,
    on=["order_id", "product_id"],
    how="left",
    validate="many_to_one"
)
df_items["returned_quantity"] = df_items["returned_quantity"].fillna(0).astype(int)
df_items["purchased_quantity"] = df_items["quantity"].astype(int)
df_items = df_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)
df_items["order_date_key"] = (
    pd.to_datetime(df_items["order_date"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)
df_items = df_items.merge(
    dim_geography,
    on="zip",
    how="left",
    validate="many_to_one"
).merge(
    dim_product,
    on="product_id",
    how="left",
    validate="many_to_one"
)
df_items["item_key"] = df_items.index + 1
df_items["item_id"] = df_items["item_id"].astype(str)
df_items["order_id"] = df_items["order_id"].astype(str)
fact_cols = [
    "item_key",
    "item_id",
    "order_id",
    "order_date_key",
    "geography_key",
    "product_key",
    "purchased_quantity",
    "returned_quantity"
]
df_fact_items = df_items[fact_cols]
df_fact_items.info()

C:\Users\Tris\AppData\Local\Temp\ipykernel_11712\932810638.py:1: DtypeWarning: Columns (6: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(


<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   item_key            714669 non-null  int64
 1   item_id             714669 non-null  str  
 2   order_id            714669 non-null  str  
 3   order_date_key      714669 non-null  int64
 4   geography_key       714669 non-null  int64
 5   product_key         714669 non-null  int64
 6   purchased_quantity  714669 non-null  int64
 7   returned_quantity   714669 non-null  int64
dtypes: int64(6), str(2)
memory usage: 43.6 MB


### Nạp dữ liệu (Load) vào Microsoft SQL Server theo Batch


In [3]:
df_fact_items.to_sql(
    name='fact_order_items',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=10000
)
print(f"Đã nạp thành công {len(df_fact_items):,} dòng vào FACT_ORDER_ITEMS trên SQL Server!")


Đã nạp thành công 714,669 dòng vào FACT_ORDER_ITEMS trên SQL Server!
